In [2]:
!pip install pyreadstat

   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.0 MB ? eta -:--:--
   ---------- ----------------------------- 0.5/2.0 MB 1.1 MB/s eta 0:00:02
   -------------------- ------------------- 1.0/2.0 MB 1.4 MB/s eta 0:00:01
   ------------------------- -------------- 1.3/2.0 MB 1.5 MB/s eta 0:00:01
   ------------------------------- -------- 1.6/2.0 MB 1.5 MB/s eta 0:00:01
   ---------------------------------------- 2.0/2.0 MB 1.5 MB/s  0:00:01

  Attempting uninstall: narwhals

    Found existing installation: narwhals 2.7.0

   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
    Uninstalling narwhals-2

In [2]:
!pip install chembl_webresource_client

In [3]:
import pandas as pd
from chembl_webresource_client.new_client import new_client

# 1. Connect to the ChEMBL target API
target = new_client.target
target_query = target.search('EGFR')
targets = pd.DataFrame.from_dict(target_query)

# 2. Select the specific human EGFR protein target ID
selected_target = 'CHEMBL203'

# 3. Fetch all bioactivity data for this target that reports IC50 values
bioactivity = new_client.bioactivity
res = bioactivity.filter(target_chembl_id=selected_target).filter(standard_type="IC50")
df = pd.DataFrame.from_dict(res)

# 4. Save this raw data as a CSV file in your folder
df.to_csv('EGFR_raw_bioactivity_data.csv', index=False)
print(f"Success! Data downloaded. Total rows of data: {len(df)}")

D:\New folder\Lib\site-packages\chembl_webresource_client\__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __version__ = __import__('pkg_resources').get_distribution('chembl_webresource_client').version


AttributeError: 'NewClient' object has no attribute 'bioactivity'

In [4]:
import pandas as pd
from chembl_webresource_client.new_client import new_client

# 1. Target ID for human EGFR protein
selected_target = 'CHEMBL203'

# 2. Use the correct activity endpoint to fetch data
# We use new_client.activity instead of bioactivity
activities = new_client.activity
res = activities.filter(target_chembl_id=selected_target).filter(standard_type="IC50")

# 3. Convert the results into a data frame
df = pd.DataFrame.from_dict(res)

# 4. Save this raw data as a CSV file in your folder
df.to_csv('EGFR_raw_bioactivity_data.csv', index=False)
print(f"Success! Data downloaded. Total rows of data: {len(df)}")

Success! Data downloaded. Total rows of data: 26600


In [5]:
import pandas as pd

# 1. Load the raw data we just downloaded
df = pd.read_csv('EGFR_raw_bioactivity_data.csv')

# 2. Drop rows where 'standard_value' or 'canonical_smiles' is blank/missing
df_clean = df.dropna(subset=['standard_value', 'canonical_smiles'])

# 3. Drop duplicate chemical structures to keep the data unique
df_clean = df_clean.drop_duplicates(subset=['canonical_smiles'])

# 4. Keep only the columns we need for our research paper
# molecule_chembl_id = The ID number of the drug
# canonical_smiles = The chemical layout code
# standard_value = The IC50 number (potency)
columns_to_keep = ['molecule_chembl_id', 'canonical_smiles', 'standard_value']
df_final = df_clean[columns_to_keep]

# 5. Save this clean dataset as a new file
df_final.to_csv('EGFR_processed_bioactivity_data.csv', index=False)

print(f"Data Cleaning Complete!")
print(f"Original Rows: {len(df)}")
print(f"Cleaned Rows Remaining: {len(df_final)}")

C:\Users\osaid\AppData\Local\Temp\ipykernel_30132\1183219890.py:4: DtypeWarning: Columns (19) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('EGFR_raw_bioactivity_data.csv')


Data Cleaning Complete!
Original Rows: 26600
Cleaned Rows Remaining: 13660


In [6]:
import csv

# 1. Read the processed data and label it
input_filename = 'EGFR_processed_bioactivity_data.csv'
output_filename = 'EGFR_labeled_bioactivity_data.csv'

active_count = 0
inactive_count = 0
intermediate_count = 0
rows_to_write = []

with open(input_filename, mode='r', newline='', encoding='utf-8') as infile:
    reader = csv.DictReader(infile)
    # Get the original column names and add our new column name
    fieldnames = reader.fieldnames + ['bioactivity_class']
    
    for row in reader:
        try:
            # Convert standard_value to a decimal number
            value = float(row['standard_value'])
            
            # Apply our classification rules
            if value <= 1000:
                bioactivity_class = "active"
                active_count += 1
            elif value >= 10000:
                bioactivity_class = "inactive"
                inactive_count += 1
            else:
                bioactivity_class = "intermediate"
                intermediate_count += 1
        except ValueError:
            # If a row has a weird formatting issue, default to intermediate
            bioactivity_class = "intermediate"
            intermediate_count += 1
            
        row['bioactivity_class'] = bioactivity_class
        rows_to_write.append(row)

# 2. Save everything to our new file
with open(output_filename, mode='w', newline='', encoding='utf-8') as outfile:
    writer = csv.DictWriter(outfile, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows_to_write)

print("Data Labeling Complete!")
print(f"active:       {active_count}")
print(f"inactive:     {inactive_count}")
print(f"intermediate: {intermediate_count}")

Data Labeling Complete!
active:       9416
inactive:     2363
intermediate: 1881
